![WEkEO by Copernicus and Partners](https://github.com/wekeo/frameworks/blob/main/img/WEkEO_banner_EUMETSAT.png?raw=true)

Copyright (c) 2025 European Union 

License: MIT

# How to use the WEkEO Harmonised Data Access (HDA) API

**This notebook can be run on the WEkEO JupyterHub. You can also view it on GitHub.**

[![launch-WEKEO](https://img.shields.io/badge/launch-WEKEO-1a4696.svg)](https://jupyterhub.prod.wekeo2.eu/hub/user-redirect/lab/tree/public/wekeo4data/wekeo-hda/wekeo_harmonised_data_access_api.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/wekeo/wekeo-hda/blob/master/wekeo_harmonised_data_access_api.ipynb)

**PREREQUISITES**

This notebook has the following prerequisites:
- [A WEkEO account](https://my.wekeo.eu/user-registration) if you are using or plan to use WEkEO.

**LEARNING OUTCOMES**

At the end of this notebook you will know:

- how to import the WEkEO HDA client
- how to set up your WEkEO credentials for the HDA API
- how to find a dataset and build a data request in the WEkEO Data Viewer
- how to search for and download data using the HDA API

**CONTENTS**

1. [Import the WEkEO HDA client](#1-import-the-wekeo-hda-client)  
2. [Configure the HDA API authentication](#2-configure-the-hda-api-authentication)  
3. [Search for datasets on WEkEO](#3-search-for-datasets-on-wekeo)  
4. [Create the data request](#4-create-the-data-request)  
5. [Download data](#5-download-data)  
6. [Take home messages](#6-take-home-messages)

<hr>

[WEkEO](https://wekeo.copernicus.eu/) is the EU Copernicus DIAS (Data and Information Access Service) reference service for environmental data, virtual processing environments and skilled user support. WEkEO offers access to a variety of data, including all Copernicus services and a multitude of parameters sensored from Sentinel-1, Sentinel-2,  Sentinel-3, Sentinel-5P and SPOT satellites family. It further offers access to climate reanalysis and seasonal forecast data.

The [Harmonised Data Access (HDA) API](https://help.wekeo.eu/en/articles/9515753-what-is-the-harmonized-data-access-hda-api), a REST interface, allows users to subset and download datasets from WEkEO.

This notebook is a step-by-step guide on how to search for and download data from WEkEO using the `HDA API`.

## 1. Import the WEkEO HDA client

The `hda` package provides a fully compliant Python 3 client that can be used to search and download products using the WEkEO Harmonized Data Access API.  
The HDA is a RESTful interface allowing users to search and download WEkEO datasets.
For more information, please check the [official documentation](https://hda.readthedocs.io/en/latest/index.html) and our [Help Center](https://help.wekeo.eu/en/collections/3530725-wekeo-data-download).

If you are working in the WEkEO JupyterLab, the **HDA client is already installed in the `wekeolab` kernel**, so make sure it is selected on the top right and simply run the following cell:

In [1]:
from hda import Client, Configuration
import os
import getpass
import hda
from pathlib import Path

*Note: if you're working in a local environment, you can install the `hda` package easily via PyPI or Mamba/Conda as shown [in this article](https://help.wekeo.eu/en/articles/6751608-how-to-use-the-hda-api-in-python#h_a309ecb279).*

## 2. Configure the HDA API authentication

In order to download data using the HDA API, you need to provide your WEkEO credentials. You can do this in two ways:

* **Option 1**: by creating a credentials file (*recommended, to be done only once*)
* **Option 2**: by supplying your credentials directly in the script (*not recommended, but sometimes useful*)

### Option 1: create a credentials file (only once)

To download data from the [WEkEO Data Catalogue](https://wekeo.copernicus.eu/data?view=catalogue), we will use the WEkEO [Harmonised Data Access (HDA) adaptor](https://help.wekeo.eu/en/articles/9515753-what-is-the-harmonized-data-access-hda-api).  
If you are currently working on the WEkEO JupyterHub, this client should already be installed. Otherwise, you can use the environment file included in this repository (`environment.yml`) to build your own Python environment (as detailed in the README) and install it. You are also welcome to view the [source code](https://github.com/ecmwf/hda) for more information and further instructions.

In order to download data using the WEkEO HDA adaptor, we need to provide our credentials. To do this, create a file called `.hdarc` in your home directory. For most systems, the home directory can be:

- `\user\username`
- `/users/username`
- `/home/username`

depending on your operating system. In this file, add the following information exactly as shown:


`user:<your_user_name>`

`password:<your_password>`

You must replace `<your_user_name>` and `<your_password>` with your WEkEO account information.  
If you don't have an account yet, please register at [https://www.wekeo.eu/](https://www.wekeo.eu/).  

Once these credentials are entered, the `hda` client will automatically read them from the file when you use it.

Lets create our credentials file, if it doesn't already exist.

In [ ]:
 # load credentials
wekeo_credentials_file = Path(Path.home() / '.hdarc' )

def get_credentials():
    user = input('Enter your WEkEO user name: ').strip()
    password = getpass.getpass('Enter your WEkEO password: ').strip()
    return user, password

def save_credentials(user, password):
    os.makedirs(os.path.dirname(wekeo_credentials_file), exist_ok=True)
    with open(wekeo_credentials_file, "w", encoding="utf-8") as f:
        f.write(f'user: {user}\npassword: {password}')

# Try loading saved credentials
if not wekeo_credentials_file.is_file():
    user, password = get_credentials()
    save_credentials(user, password)

# Try to get token, retry if credentials are invalid
while True:
    try:
        token = hda.Client().token
        print(f"Your WEkEO token: '{token}'")
        break  # Exit loop if successful
    except hda.api.ConfigurationError as e:
        print("Invalid username or password. Please try again.")
        print (e)
        user, password = get_credentials()
        save_credentials(user, password)

Now the HDA will recognize your WEkEO credentials automatically, and you can simply call the HDA client as follows:

In [3]:
hda_client = Client()

### Option 2: provide credentials directly (every time)

You can provide your credentials directly as follows, replacing both username and password:

In [ ]:
conf = Configuration(user="username", password="password")
hda_client = Client(config=conf)

*Note: this method is convenient in the short term, but is not really recommended as you have to put your username and password in this notebook, and run the risk of accidentally sharing them. This method also requires you to authenticate on a notebook-by-notebook basis.*

## 3. Search for datasets on WEkEO

Once on the WEkEO [Expert Data Viewer](https://wekeo.copernicus.eu/data?view=viewer), click on the "+" icon from the Layers tab to open the Catalogue and search for your dataset of interest. You can use the free-text search bar (e.g. *`sentinel-3`*) and browse through the results. You can also use the filters on the left to refine your search and look by satellite plaform, sensor, Copernicus service, area (region of interest), as well as through a variety of flags.

<img src='./img/WEkEO_catalogue.png' width='80%'></img>

You can click on **Details** for the dataset you are interested in to open the dataset's page, including a description, temporal and spatial extents, and metadata.  
For the purpose of this notebook, we will take the dataset `OLCI Level 1B Full Resolution - Sentinel-3`:

<img src='./img/Sentinel_3_datasetID.png' width='50%'></img>

The dataset's page provides the following information:
- **Abstract**: containing a general description of the dataset
- **Classification**: including the datasetID 
- **Resources**: such as a link to the Product Data Format Specification guide, and JSON metadata
- **Contacts**: where you can find further information about the data source from its provider

Click on **Add to map** to add the dataset to the Layers and visualize data on the map.  

## 4. Create the data request

The HDA API reads the data query from a JSON-like dictionary. Depending on the selected dataset, certain parameters are mandatory.  
To avoid mistakes, it is **highly recommended to build the query from the [Expert Data Viewer](https://wekeo.copernicus.eu/data?view=viewer)**.

From the Layers tab, you can click on the **Subset and download** button to build the data request via Graphical User Interface:

<img src='./img/download_button.png' width='20%'></img>

Once you defined the download parameters, you can `(1)` click on **Show API request(s)** and `(2)` on **Copy**:

<img src='./img/api_request.png' width='60%' />


Then you can paste the request in a variable as follows:

In [4]:
query = {
  "dataset_id": "EO:EUM:DAT:SENTINEL-3:OL_1_EFR___",
  "startdate": "2025-05-12T00:00:00.000Z",
  "enddate": "2025-05-14T23:59:59.999Z",
  "bbox": [
    1.3287978665692197,
    45.538598473683436,
    5.158722327438545,
    47.78018177390331
  ],
  "type": "OL_1_EFR___",
  "timeliness": "NT",
  "itemsPerPage": 200,
  "startIndex": 0
}

The following cell is used to perform the search based on these parameters, and the results are stored in the `matches` variable:

In [5]:
# Ask the result for the query passed in parameter
matches = hda_client.search(query)

# List the results
print(matches)

SearchResults[items=5,volume=4.4GB]


## 5. Download data

As a final step, you simply need to download the results of the `matches` variable. Choose only the first results with `matches[0]` to save some downloading time. 

In [6]:
matches[0].download()

*Note: you can check [our article](https://help.wekeo.eu/en/articles/6751608-how-to-use-the-hda-api-in-python#h_80a0396d92) to download specific files of the results.*

## 6. Take home messages

- The `hda` Python client gives you programmatic access to the WEkEO datasets through the Harmonised Data Access (HDA) API.
- Store your WEkEO credentials once in the `.hdarc` file in your home directory; the client then reads them automatically.
- The easiest way to build a data request is to copy it from **Show API request** in the WEkEO Data Viewer, and then `search()` and `download()` it with the client.

<hr>

[View on GitHub](https://github.com/wekeo) | 
[Visit WEkEO](https://wekeo.copernicus.eu) | [Run Notebooks on the WEkEO Workspace](https://jupyterhub.prod.wekeo2.eu/) | 
[Contact helpdesk for support](mailto:support@wekeo.eu)